In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interact_manual
from copy import deepcopy
from tqdm.notebook import tqdm
import networkx as nx


In [2]:
# given topic number, assign proper probability according to zipf's distribution with alpha = 1
def zipf_prob(topic_num):
    prob = [1/(i+1) for i in range(topic_num)]
    # get normalized probability
    prob = [i/sum(prob) for i in prob]
    return prob

In [21]:
class topic_space:
    def __init__(self, dim, topic_num):
        self.dim = dim
        self.topic_num = topic_num
        self.topic_embeddings = np.random.rand(topic_num, dim)
        # sample topic frequency from a zipf distribution
        self.topic_frequency = zipf_prob(topic_num)
        self.dist = np.zeros((self.topic_num, self.topic_num))
        self.update_all_dist()
        
    def get_dist(self):
        return self.dist
    
    def get_inv_norm_dist(self):
        # return normalized inverse distance matrix
        inv_dist = 1 / self.dist
        np.fill_diagonal(inv_dist, 0)
        return inv_dist / inv_dist.sum(axis=1, keepdims=True)
        
    def get_topic_num(self):
        return self.topic_num
    
    def get_dim(self):
        return self.dim
    
    def get_topic_frequency(self):
        return self.topic_frequency
    
    def get_topic_embeddings(self):
        return self.topic_embeddings
    
    def copy_from(self, topic_space):
        self.topic_embeddings = topic_space.get_topic_embeddings()
        self.topic_frequency = topic_space.get_topic_frequency()
        self.update_all_dist()
    
    # embeddings
    
    def perturb_topic_embeddings(self, perturbation):
        assert perturbation.shape == self.topic_embeddings.shape
        self.topic_embeddings += perturbation
        self.update_all_dist()
        
    def perturb_topic_embeddings_gaussian(self, std):
        perturbation = np.random.normal(0, std, self.topic_embeddings.shape)
        self.perturb_topic_embeddings(perturbation)
        
    # dist    
        
    def update_all_dist(self):
        # calculate pairwise distances between topic embeddings
        for i in range(self.topic_num):
            for j in range(self.topic_num):
                self.dist[i, j] = np.linalg.norm(self.topic_embeddings[i] - self.topic_embeddings[j])
                
    def update_dist(self, topic_id1, topic_id2):
        self.dist[topic_id1, topic_id2] = np.linalg.norm(self.topic_embeddings[topic_id1] - self.topic_embeddings[topic_id2])
        self.dist[topic_id2, topic_id1] = self.dist[topic_id1, topic_id2]
        
    def calculate_pairwise_dist(self, events):
        # calculate the sum of all pairwise distances between topics in an event, according to the dist matrix
        dist_list = []
        for event in events:
            dist = 0
            # make a list of all pairs of topics in the event
            pairs = [(a, b) for idx, a in enumerate(event) for b in event[idx + 1:]]
            dist_list.append(sum([self.dist[pair] for pair in pairs]))
        return dist_list
  
    # Event generation
    
    def generate_events(self, event_num, event_topic_num):
        events = np.zeros((event_num, event_topic_num)).astype(int)
        for i in range(event_num):
            # first, sample event according to its frequency
            events[i][0] = np.random.choice(self.topic_num, p=self.topic_frequency)
            # next, sample j-1 more topics. Let the probability of sampling topic j be proportional to the inverse of sum of dist between topic j and all other topics already sampled, except themselves.
            for j in range(1, event_topic_num):
                dist = self.dist[events[i, :j], :]
                dist = np.sum(dist, axis=0)
                dist = 1 / dist
                dist[events[i, :j]] = 0
                dist = dist / np.sum(dist)
                events[i, j] = np.random.choice(self.topic_num, p=dist)
        
        return events

    # Activity generation
    
    def activity_graph_generation(self):
        activity_graph = nx.Graph()
        activity_graph.add_nodes_from(range(self.topic_num))

        nx.set_node_attributes(activity_graph, 0, 'visited')
        for i in range(self.topic_num):
            for j in range(self.topic_num):
                activity_graph.add_edge(i, j, passed=0)
                
        return activity_graph
        
    def activity_generation(self, events, step_num):
        topic_num = self.topic_num
        
        events_topic_sum = np.zeros(topic_num).astype(int)
        for event in events:
            for topic in event:
                events_topic_sum[topic] += 1
        
        comm_inv_norm_dist = self.get_inv_norm_dist()
        activity_graph = self.activity_graph_generation()

        current_state = events_topic_sum
        for i in current_state.nonzero()[0]:
            activity_graph.nodes[i]['visited'] += 1
        next_state = np.zeros_like(events_topic_sum)

        for r in range(step_num):
            events_topic_sum.nonzero()
            current_loc = current_state.nonzero()[0]
            for loc in current_loc:
                next_move = np.random.choice(a=topic_num, size=current_state[loc], p=comm_inv_norm_dist[loc])
                for i in next_move:
                    next_state[i] += 1
                    activity_graph.nodes[i]['visited'] += 1
                    activity_graph.edges[loc, i]['passed'] += 1
                    
            current_state = next_state
            next_state = np.zeros_like(events_topic_sum)
                        
        return activity_graph

### 1. Generate topic spaces

In [22]:
def generate_topic_spaces(dim, topic_num, comm_num, std):
    
    # generate general topic space
    general = topic_space(dim=dim, topic_num=topic_num)
    community_list = []
    
    # generate community topic spaces and perturb their embeddings from general topic space
    for i in range(comm_num):
        community_list.append(topic_space(dim=dim, topic_num=topic_num))
        community_list[i].copy_from(general)
        community_list[i].perturb_topic_embeddings_gaussian(std=std)
        
    return general, community_list

In [23]:
dim = 16
topic_num = 100
std = 0.1
comm_num = 5

general, community_list = generate_topic_spaces(dim=dim, topic_num=topic_num, comm_num=comm_num, std=std)

### 2. Event generation & filter

In [24]:
def event_generation(general, community_list, event_num, event_topic_num, prob_func):
    
    # generate events
    events = general.generate_events(event_num=event_num, event_topic_num=event_topic_num)
    
    # calculate pairwise distances between topics in events at general / community topic spaces
    general_dist = general.calculate_pairwise_dist(events)
    comm_dist_list = [community_list[i].calculate_pairwise_dist(events) for i in range(comm_num)]
    
    # calculate the difference of distances between general / community topic spaces of each events
    dist_diff_list = np.array(general_dist)-np.array(comm_dist_list)
    
    #define probability of accepting each event according to a distance and custom probability function
    prob_list = myprob(dist_diff_list)
    
    return events, np.random.binomial(1, prob_list)
    

def myprob(x, a=1):
    # f(0) = 0.5, f(-inf) = 0, f(inf) = 1
    # a = 1: steepness of the curve
    return 0.5 + 0.5 * np.tanh(a*x)


In [25]:
event_num = 1000
event_topic_num = 3
prob_func = myprob

events, event_filter = event_generation(general=general, community_list=community_list, event_num=event_num, event_topic_num=event_topic_num, prob_func=myprob)

C:\Users\Seungwoong\AppData\Local\Temp\ipykernel_168952\360853766.py:81: RuntimeWarning: divide by zero encountered in divide
  dist = 1 / dist


### 3. Activity generation

In [26]:
step_num = 10
activity_graph_list = []

for i, community in enumerate(community_list):
    events_eff = events[event_filter[i].nonzero()[0]]
    activity_graph_list.append(community.activity_generation(events=events_eff, step_num=step_num))

C:\Users\Seungwoong\AppData\Local\Temp\ipykernel_168952\360853766.py:16: RuntimeWarning: divide by zero encountered in divide
  inv_dist = 1 / self.dist


### 4. Topic embeddings / frequency change

### 5. Main

In [28]:
dim = 16
topic_num = 100
std = 0.1
comm_num = 5
event_num = 1000
event_topic_num = 3
prob_func = myprob
step_num = 10
timestep = 20


general, community_list = generate_topic_spaces(dim=dim, topic_num=topic_num, comm_num=comm_num, std=std)
activity_graph_list = []
for t in tqdm(range(timestep)):
    events, event_filter = event_generation(general=general, community_list=community_list, event_num=event_num, event_topic_num=event_topic_num, prob_func=myprob)
    activity_graph_list.append([])

    for i, community in enumerate(community_list):
        events_eff = events[event_filter[i].nonzero()[0]]
        activity_graph_list[-1].append(community.activity_generation(events=events_eff, step_num=step_num))

  0%|          | 0/20 [00:00<?, ?it/s]

C:\Users\Seungwoong\AppData\Local\Temp\ipykernel_168952\360853766.py:81: RuntimeWarning: divide by zero encountered in divide
  dist = 1 / dist
C:\Users\Seungwoong\AppData\Local\Temp\ipykernel_168952\360853766.py:16: RuntimeWarning: divide by zero encountered in divide
  inv_dist = 1 / self.dist
